# 02 · Citation-relation labels and the fine-tuned SciBERT classifier (component A)

**Purpose.** Label mined citation contexts with the configured LLM against a seven-class relation taxonomy, measure the LLM against an adjudicated gold set, then fine-tune SciBERT and compare it with a majority-class, a TF-IDF+LR and a zero-shot LLM baseline.

| | |
|---|---|
| **Inputs** | `data/citation_contexts.jsonl` (notebook 01) |
| **Outputs** | `data/llm_labels.jsonl`, `data/gold_blind.jsonl`, `models/scibert_relations/`, `results/stage2_labelling.json`, `results/stage2_scibert.json`, `results/stage2_benchmark.json`, `results/stage2_gold_agreement.json` |
| **Expected runtime (Colab T4)** | labelling 1-3 h (API-bound); SciBERT ~15 min; baselines ~10 min |
| **Required secrets** | `GROQ_API_KEY` (or `GEMINI_API_KEY`) |
| **GPU** | T4 recommended for SciBERT and the zero-shot baseline |

Everything is cached under `/content/drive/MyDrive/litreview/` and every stage is
**resumable**: if Colab disconnects, re-run all cells and it continues where it
stopped. Only the cell marked CONFIG needs editing.


In [ ]:
# --- Colab setup: Drive, project code, secrets. Safe to re-run. ---
import json, os, subprocess, sys, zipfile
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive/litreview" if ON_COLAB else os.path.abspath("..")
os.makedirs(ROOT, exist_ok=True)
os.environ["LITREVIEW_ROOT"] = ROOT
REPO = os.path.join(ROOT, "repo")

# Optional: a git URL to clone the project from. Leave blank to use the folder or
# the zip on Drive instead.
REPO_URL = ""

def _find_project() -> str | None:
    """Locate the project code: an existing folder, a zip on Drive, or a clone."""
    if os.path.exists(os.path.join(REPO, "pyproject.toml")):
        return REPO
    # a litreview_*.zip uploaded anywhere under MyDrive/litreview
    zips = sorted(Path(ROOT).glob("litreview*.zip")) + sorted(Path(ROOT).glob("**/litreview*.zip"))
    for z in zips:
        print(f"extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(ROOT)
        # the zip may contain a top-level folder; find the pyproject inside
        for cand in [REPO, ROOT] + [str(p.parent) for p in Path(ROOT).glob("*/pyproject.toml")]:
            if os.path.exists(os.path.join(cand, "pyproject.toml")):
                return cand
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=False)
        if os.path.exists(os.path.join(REPO, "pyproject.toml")):
            return REPO
    return None

if ON_COLAB:
    found = _find_project()
    if not found:
        raise SystemExit(
            "Could not find the project code. Do one of these, then re-run this cell:
"
            "  1. upload litreview_colab.zip into MyDrive/litreview/ (easiest), or
"
            "  2. copy the project folder to MyDrive/litreview/repo, or
"
            "  3. set REPO_URL above to a git URL.
"
            "The notebooks deliberately contain no second copy of the code.")
    REPO = found
    print("project code:", REPO)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[ml,eval]"],
                   check=True)

import litreview
from litreview.config import SETTINGS, paths, secret
P = paths().mkdirs()
print("data root:", P.root)
print("already present:", {f.name: f.stat().st_size // 1024 for f in sorted(P.data.glob("*.jsonl"))})


In [ ]:
# ======================= CONFIG (the only cell to edit) =======================
PROVIDER   = "groq"          # groq | gemini | ollama
MODEL      = ""              # "" uses the provider's default from litreview/config.py
N_LABEL    = 4000            # contexts to label this session (append-only, resumable)
BATCH_SIZE = 8               # contexts per request: 8x saving on the request budget
IN_RW_ONLY = True            # label citations inside related-work sections only
N_GOLD     = 200             # size of the adjudicated gold set
EPOCHS, LR, TRAIN_BATCH = 4, 2e-5, 16
USE_SEED_CORPORA = True      # add SciCite + ACL-ARC rows, mapped onto our taxonomy
SEED = 0
# ==============================================================================


In [ ]:
# Provider comes from one config. Keys are read from Colab secrets, never hard-coded.
import os
os.environ["LITREVIEW_PROVIDER"] = PROVIDER
if MODEL:
    os.environ["LITREVIEW_MODEL"] = MODEL

from litreview.config import Settings
from litreview.llm import install_call_logger, probe, usage_summary
import litreview.config as _cfg
_cfg.SETTINGS = Settings()                      # re-read after setting the env vars
import litreview.llm as _llm
_llm.SETTINGS = _cfg.SETTINGS

install_call_logger(P.results / "llm_calls.jsonl")
print(json.dumps(_cfg.SETTINGS.describe(), indent=1))
print(json.dumps(probe(_cfg.SETTINGS), indent=1))


### The taxonomy

Seven labels with one definition each and an ordered set of tie-break rules. The
definitions live in `litreview.relations.taxonomy` so the LLM prompt, the human
labelling guide (`docs/labelling_guide.md`) and the training code cannot disagree
about what a label means.

In [ ]:
from litreview.relations.taxonomy import (ACL_ARC_LIMITS, ACL_ARC_MAP, DEFINITIONS,
                                           DECISION_RULES, SCICITE_LIMITS, SCICITE_MAP)
for k, v in DEFINITIONS.items():
    print(f"{k:18s} {v}")
print("\nTie-break order:")
for i, r in enumerate(DECISION_RULES, 1):
    print(f"  {i}. {r}")
print("\nSciCite mapping:", SCICITE_MAP, "\n  limits:", SCICITE_LIMITS)
print("\nACL-ARC mapping:", ACL_ARC_MAP, "\n  limits:", ACL_ARC_LIMITS)


### Label with the LLM

Append-only and keyed by a hash of (citing paper, cite key, sentence), so a daily
rate limit is not a failure: re-run this cell tomorrow and it continues. Each
request carries `BATCH_SIZE` contexts and asks for a label, a short rationale that
must quote the cue words, and a confidence.

In [ ]:
import random
from tqdm.auto import tqdm
from litreview.relations.label import label_contexts, load_labels
from litreview.relations.taxonomy import LABELS

rows = []
for line in (P.data / "citation_contexts.jsonl").open():
    try:
        r = json.loads(line)
    except json.JSONDecodeError:
        continue
    if r.get("split") != "train":
        continue
    if IN_RW_ONLY and not r.get("in_related_work"):
        continue
    if len(r.get("cite_sentence") or "") < 40:
        continue
    rows.append(r)
random.Random(SEED).shuffle(rows)
print(f"{len(rows)} candidate contexts")

stats = label_contexts(rows, P.data / "llm_labels.jsonl", batch_size=BATCH_SIZE,
                       limit=N_LABEL, progress=lambda b: tqdm(b, desc="label"))
labelled = load_labels(P.data / "llm_labels.jsonl")
rep = {"stats": stats, "n_labelled_total": len(labelled),
       "label_distribution": {l: sum(1 for r in labelled if r["llm_label"] == l) for l in LABELS},
       "provider": _cfg.SETTINGS.describe(), "usage": usage_summary()}
(P.results / "stage2_labelling.json").write_text(json.dumps(rep, indent=2))
print(json.dumps({k: v for k, v in rep.items() if k != "provider"}, indent=2))


### The adjudicated gold set

200 contexts, stratified over the LLM's predicted label so the rare relations are
represented at all. The sampler writes a **blind** file with no LLM label in it;
that is what keeps the gold labels independent of the labeller. Label `gold_blind.jsonl` by hand
(fill in `human_label`), save it as `gold_labelled.jsonl`, then run the scoring
cell.

Because the sample is stratified, the resulting agreement is **not** an estimate of
accuracy on a uniformly random citation context, and the report says so.

In [ ]:
import subprocess, sys
SCRIPTS = (Path(REPO) / "scripts") if ON_COLAB else Path("../scripts")
subprocess.run([sys.executable, str(SCRIPTS / "sample_gold.py"), "--n", str(N_GOLD)], check=False)
print("\nNow hand-label", P.data / "gold_blind.jsonl",
      "\n -> save as", P.data / "gold_labelled.jsonl", "with human_label filled in.")


In [ ]:
# Run once gold_labelled.jsonl exists.
if (P.data / "gold_labelled.jsonl").exists():
    subprocess.run([sys.executable, str(SCRIPTS / "score_gold.py")], check=False)
else:
    print("gold_labelled.jsonl not present yet; skipping agreement scoring")


### Fine-tune SciBERT

Input: the citation context with the target citation marked `[[CITED]]` and any
other citations in the sentence collapsed to `[OTHER]`, paired with the cited
paper's title as a second segment. Class weights are inverse-frequency because
`background` dominates. The dev split is taken **by citing paper**, so no paper
appears on both sides. Gold-set rows are excluded from training.

In [ ]:
from litreview.relations import seed_data
from litreview.relations.scibert import RelationClassifier, split_by_paper, train

rows = [{**r, "label": r["llm_label"]} for r in load_labels(P.data / "llm_labels.jsonl")]
gold = []
if (P.data / "gold_labelled.jsonl").exists():
    gold = [{**json.loads(l), "label": json.loads(l)["human_label"]}
            for l in (P.data / "gold_labelled.jsonl").open()
            if json.loads(l).get("human_label") in LABELS]
gold_ids = {r["context_id"] for r in gold}
rows = [r for r in rows if r["context_id"] not in gold_ids]

extra = {}
if USE_SEED_CORPORA:
    sc = seed_data.load_scicite(P.cache, limit=2000)
    aa = seed_data.load_acl_arc(P.cache, limit=2000)
    extra = {"scicite": {k: v for k, v in sc.items() if k != "rows"},
             "acl_arc": {k: v for k, v in aa.items() if k != "rows"}}
    rows += sc.get("rows", []) + aa.get("rows", [])
    print(json.dumps(extra, indent=1))

tr, dev = split_by_paper(rows, frac_dev=0.15, seed=SEED)
print(f"train {len(tr)} rows / dev {len(dev)} rows")
report = train(tr, P.models / "scibert_relations", epochs=EPOCHS, lr=LR,
               batch_size=TRAIN_BATCH, dev_rows=dev, seed=SEED,
               progress=lambda dl, d: tqdm(dl, desc=d))
report["seed_corpora"] = extra
if gold:
    report["gold_eval"] = RelationClassifier(P.models / "scibert_relations").evaluate(gold)
    print("gold macro-F1", report["gold_eval"]["macro_f1"],
          "accuracy", report["gold_eval"]["accuracy"])
(P.results / "stage2_scibert.json").write_text(json.dumps(report, indent=2))
print(json.dumps({k: v for k, v in report.items()
                  if k not in ("gold_eval", "seed_corpora")}, indent=2))


### Baselines

Majority class, TF-IDF + logistic regression, and a zero-shot LLM using the same
prompt as the labeller. All four systems are scored on the same adjudicated gold set.

In [ ]:
subprocess.run([sys.executable, str(SCRIPTS / "run_stage2.py"), "bench"], check=False)
if (P.results / "stage2_benchmark.json").exists():
    b = json.loads((P.results / "stage2_benchmark.json").read_text())
    for k in ("majority", "tfidf_lr", "zero_shot_llm", "scibert"):
        if k in b:
            print(f"{k:16s} acc={b[k]['accuracy']:.3f}  macroF1={b[k]['macro_f1']:.3f}")


In [ ]:
# Expose the classifier the later notebooks use.
from litreview.relations.scibert import RelationClassifier
clf = RelationClassifier(P.models / "scibert_relations")
print(clf.classify("Unlike [[CITED]], which assumes a single reference summary, we ...",
                   "Evaluating the factual consistency of abstractive text summarization"))
print(json.dumps(usage_summary()["overall"], indent=1))


### Next

**03_crew_extract_verify.ipynb** — the Extractor agent and the NLI-based Verifier,
whose thresholds are fitted on a hand-checked dev set.